<a href="https://colab.research.google.com/github/Shane-Boulter/MyNFL/blob/data_store/conversation/data-store-status-checker/data_store_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vertex AI Search and Conversation Data Store Status Checker

_Using Google Cloud Discovery Engine APIs for Vertex AI Search and Conversation_

<table align="left">

  <td>
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/conversation/data-store-status-checker/data_store_checker.ipynb">
      <img src="https://cloud.google.com/ml-engine/images/colab-logo-32px.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/conversation/data-store-status-checker/data_store_checker.ipynb">
      <img src="https://cloud.google.com/ml-engine/images/github-logo-32px.png" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/conversation/data-store-status-checker/data_store_checker.ipynb">
      <img src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" alt="Vertex AI logo">
      Open in Vertex AI Workbench
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/conversation/data-store-status-checker/data_store_checker.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/conversation/data-store-status-checker/data_store_checker.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/conversation/data-store-status-checker/data_store_checker.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/conversation/data-store-status-checker/data_store_checker.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/conversation/data-store-status-checker/data_store_checker.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>            

<br><br><br>

## What is a Data Store?

A [Data Store](https://cloud.google.com/generative-ai-app-builder/docs/create-datastore-ingest) in [Vertex AI Search and Conversation](https://cloud.google.com/generative-ai-app-builder) is a collection of websites or documents, both structured and unstructured, that can be indexed for search and retrieval actions.

Data Stores are the fundamental building block behind [Vertex AI Search](https://cloud.google.com/enterprise-search) and [Vertex AI Conversation](https://cloud.google.com/generative-ai-app-builder/docs/agent-usage).

## Data Store Indexing Time

With each website or set of documents added, the Data Store needs to index the site and/or documents in order for them to be searchable. This can take up to 4 hours for new data store web content to be indexed.

Using the attached example notebook, you can query your Data Store ID to see if indexing is complete.
Once complete, you can additionally use the notebook to search your Data Store for specific pages or documents.

## Limitations

This notebook cannot be used for `Search` applications that use standard website indexing.

| | |
|-|-|
|Author(s) | [Patrick Marlow](https://github.com/kmaphoenix) |

## Objective

Simple notebook that uses the Cloud Discovery Engine API to check a Data Store for indexed docs.

---

This notebook utilizes the [`google-cloud-discoveryengine`](https://cloud.google.com/python/docs/reference/discoveryengine/latest) Python library.  
This notebook allows the user to perform the following tasks:

- ✅ Check Indexing Status of given Data Store ID.
- ✅ List all documents in a given Data Store ID.
- ✅ List all indexed URLs for a given Data Store ID
- ✅ Search all indexed URLs for a specific URL within a given Data Store ID.

---

**References:**

- [Google Cloud Discovery Engine API](https://cloud.google.com/python/docs/reference/discoveryengine/latest)

---

- Author: Patrick Marlow
- Created: 07/17/2023

---


# Install PreReqs and Authentication


In [1]:
%pip install --upgrade google-cloud-discoveryengine humanize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: humanize
    Found existing installation: humanize 4.11.0
    Uninstalling humanize-4.11.0:
      Successfully uninstalled humanize-4.11.0


In [1]:
import sys

if "google.colab" in sys.modules:
    from google.auth import default
    from google.colab import auth

    auth.authenticate_user()
    creds, _ = default()
else:
    # Otherwise, attempt to discover local credentials as described on https://cloud.google.com/docs/authentication/application-default-credentials
    pass

# Helper Methods

Run the below cell to setup the helper methods for this notebook.


In [2]:
import re
import time

from google.api_core.client_options import ClientOptions
from google.cloud import discoveryengine_v1beta as discoveryengine
import humanize


def _call_list_documents(
    project_id: str, location: str, datastore_id: str, page_token: str | None = None
) -> discoveryengine.ListDocumentsResponse:
    """Build the List Docs Request payload."""
    client_options = (
        ClientOptions(api_endpoint=f"{location}-discoveryengine.googleapis.com")
        if location != "global"
        else None
    )
    client = discoveryengine.DocumentServiceClient(client_options=client_options)

    request = discoveryengine.ListDocumentsRequest(
        parent=client.branch_path(project_id, location, datastore_id, "default_branch"),
        page_size=1000,
        page_token=page_token,
    )

    return client.list_documents(request=request)


def list_documents(
    project_id: str, location: str, datastore_id: str, rate_limit: int = 1
) -> list[discoveryengine.Document]:
    """Gets a list of docs in a datastore."""

    res = _call_list_documents(project_id, location, datastore_id)

    # setup the list with the first batch of docs
    docs = res.documents

    while res.next_page_token:
        # implement a rate_limit to prevent quota exhaustion
        time.sleep(rate_limit)

        res = _call_list_documents(
            project_id, location, datastore_id, res.next_page_token
        )
        docs.extend(res.documents)

    return docs


def list_indexed_urls(
    docs: list[discoveryengine.Document] | None = None,
    project_id: str | None = None,
    location: str | None = None,
    datastore_id: str | None = None,
) -> list[str]:
    """Get the list of docs in data store, then parse to only urls."""
    if not docs:
        docs = list_documents(project_id, location, datastore_id)
    urls = [doc.content.uri for doc in docs]

    return urls


def search_url(urls: list[str], url: str) -> None:
    """Searches a url in a list of urls."""
    for item in urls:
        if url in item:
            print(item)


def search_doc_id(
    doc_id: str,
    docs: list[discoveryengine.Document] | None = None,
    project_id: str | None = None,
    location: str | None = None,
    datastore_id: str | None = None,
) -> None:
    """Searches a doc_id in a list of docs."""
    if not docs:
        docs = list_documents(project_id, location, datastore_id)

    doc_found = False
    for doc in docs:
        if doc.parent_document_id == doc_id:
            doc_found = True
            print(doc)

    if not doc_found:
        print(f"Document not found for provided Doc ID: `{doc_id}`")


def estimate_data_store_size(
    urls: list[str] | None = None,
    docs: list[discoveryengine.Document] | None = None,
    project_id: str | None = None,
    location: str | None = None,
    datastore_id: str | None = None,
) -> None:
    """For Advanced Website Indexing data stores only."""
    if not urls:
        if not docs:
            docs = list_documents(project_id, location, datastore_id)
        urls = list_indexed_urls(docs=docs)

    # Filter to only include website urls.
    urls = list(filter(lambda x: re.search(r"https?://", x), urls))

    if not urls:
        print(
            "No urls found. Make sure this data store is for websites with advanced indexing."
        )
        return

    # For website indexing, each page is calculated as 500KB.
    size = len(urls) * 500_000
    print(f"Estimated data store size: {humanize.naturalsize(size)}")


PENDING_MESSAGE = """
No docs found.\n\nIt\'s likely one of the following issues: \n  [1] Your data store is not finished indexing. \n  [2] Your data store failed indexing. \n  [3] Your data store is for website data without advanced indexing.\n\n
If you just added your data store, it can take up to 4 hours before it will become available.
"""

# User Inputs

You can find your `datastore_id` by going following these steps:

1. Click on Vertex AI Search & Conversation
2. Select your App / Engine

![image.png](attachment:image.png)

3. Select your Available Data Store

![image-3.png](attachment:image-3.png)

4. Find your Data Store ID

![image-2.png](attachment:image-2.png)


In [3]:
project_id = "nfl-prod-453618"
location = "global"  # Options: "global", "us", "eu"
datastore_id = "nfl-support_1741889975065"

# Check Data Store Index Status

Using the `list_documents` method, we can do a check to see if the data store has finished indexing.


In [4]:
docs = list_documents(project_id, location, datastore_id)

if len(docs) == 0:
    print(PENDING_MESSAGE)
else:
    SUCCESS_MESSAGE = f"""
  Success! 🎉\n
  Your indexing is complete.\n
  Your index contains {len(docs)} documents.
  """
    print(SUCCESS_MESSAGE)


  Success! 🎉

  Your indexing is complete.

  Your index contains 247 documents.
  


# List Documents

List all the documents for a given Data Store ID


In [6]:
docs = list_documents(project_id, location, datastore_id)
for doc in docs:
  print (doc.content.uri)
#docs[0]

https://support.nfl.com/hc/en-us/articles/4989177504412-Waiver-Priority-Playoffs
https://support.nfl.com/hc/en-us/sections/4989087591196-Tickets
https://support.nfl.com/hc/en-us/articles/5403686148636-How-do-I-watch-game-replays
https://support.nfl.com/hc/en-us/articles/5403871264540-How-soon-after-the-end-of-a-game-is-the-replay-available-in-NFL
https://support.nfl.com/hc/en-us/articles/4989073286044-Game-Books
https://support.nfl.com/hc/en-us/articles/5401380067356-What-games-can-I-listen-to-with-live-audio-on-NFL
https://support.nfl.com/hc/en-us/articles/5401281634076-Who-is-Cleeng
https://support.nfl.com/hc/en-us/sections/13578182722844-Troubleshooting-Instructions
https://support.nfl.com/hc/en-us/articles/9943480596764-How-can-I-access-Sunday-Ticket-on-NFL
https://support.nfl.com/hc/en-us/articles/12227630666396-I-subscribe-to-NFL-through-the-Roku-Channel-Store-Can-I-get-the-NFL-End-of-Season-Promotion-2024
https://support.nfl.com/hc/en-us/articles/4989075106332-Why-are-my-teams-l

# Search Data Store by Doc ID

Search through all Docs in a given Data Store and find a specific Doc ID.


In [ ]:
document_id = "000a98558b6fe9aef7992c9023fb7fdb"

search_doc_id(document_id, docs=docs)

struct_data {
}
name: "projects/772105163160/locations/global/collections/default_collection/dataStores/cgc_1690226759325/branches/0/documents/000a98558b6fe9aef7992c9023fb7fdb"
id: "000a98558b6fe9aef7992c9023fb7fdb"
schema_id: "default_schema"
content {
  uri: "https://cloud.google.com/docs/security/data-loss-prevention/revoking-user-access?hl=es-419"
  mime_type: "text/html"
}
parent_document_id: "000a98558b6fe9aef7992c9023fb7fdb"



# List Indexed URLs


In [ ]:
urls = list_indexed_urls(docs=docs)
urls[0]

'https://cloud.google.com/docs/security/data-loss-prevention/revoking-user-access?hl=es-419'

# Search Indexed URLs


In [ ]:
search_url(urls, "https://cloud.google.com/docs/terraform/samples")

https://cloud.google.com/docs/terraform/samples?hl=ko
https://cloud.google.com/docs/terraform/samples?hl=fr
https://cloud.google.com/docs/terraform/samples?hl=pt-br
https://cloud.google.com/docs/terraform/samples?hl=de
https://cloud.google.com/docs/terraform/samples?hl=zh-cn
https://cloud.google.com/docs/terraform/samples?hl=ja
https://cloud.google.com/docs/terraform/samples
https://cloud.google.com/docs/terraform/samples?hl=es-419
https://cloud.google.com/docs/terraform/samples?hl=it


In [ ]:
search_url(urls, "terraform")

https://cloud.google.com/docs/terraform/getting-support?hl=de
https://cloud.google.com/docs/terraform/basic-commands?hl=ko
https://cloud.google.com/docs/terraform/policy-validation/create-cai-constraints?hl=es-419
https://cloud.google.com/docs/terraform/samples?hl=ko
https://cloud.google.com/docs/terraform/get-started-with-terraform?hl=es-419
https://cloud.google.com/docs/terraform/policy-validation/create-terraform-constraints?hl=de
https://cloud.google.com/docs/terraform/resource-management/managing-infrastructure-as-code?hl=ko
https://cloud.google.com/docs/terraform/deploy-foundation-using-terraform-from-console?hl=pt-br
https://cloud.google.com/docs/terraform/resource-management/store-state?hl=de
https://cloud.google.com/docs/terraform/policy-validation/migrate-from-terraform-validator?hl=it
https://cloud.google.com/docs/terraform/get-started-with-terraform?hl=he
https://cloud.google.com/docs/terraform/policy-validation/create-terraform-constraints?hl=he
https://cloud.google.com/do

# Estimate Data Store Size

Only for Website data stores with Advanced Website Indexing


In [ ]:
estimate_data_store_size(urls=urls)

Estimated data store size: 247.5 MB
